In [6]:
import copy
from collections import deque


backtrack_calls = 0
backtrack_failures = 0

# Helper Functions

def read_board(filename):
    board = []
    with open(filename, 'r') as f:
        for line in f:
            board.append([int(x) for x in line.strip()])
    return board


def print_board(board):
    for row in board:
        print(" ".join(str(x) for x in row))


def get_neighbors(row, col):
    neighbors = set()

    # Row & Column
    for i in range(9):
        neighbors.add((row, i))
        neighbors.add((i, col))

    # 3x3 box
    box_r, box_c = row // 3, col // 3
    for i in range(box_r * 3, box_r * 3 + 3):
        for j in range(box_c * 3, box_c * 3 + 3):
            neighbors.add((i, j))

    neighbors.remove((row, col))
    return neighbors


# CSP Initialization

def init_domains(board):
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                domains[(r, c)] = set(range(1, 10))
            else:
                domains[(r, c)] = {board[r][c]}
    return domains


# AC-3 Algorithm

def ac3(domains):
    queue = deque()

    for xi in domains:
        for xj in get_neighbors(*xi):
            queue.append((xi, xj))

    while queue:
        xi, xj = queue.popleft()
        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False
            for xk in get_neighbors(*xi):
                if xk != xj:
                    queue.append((xk, xi))
    return True


def revise(domains, xi, xj):
    revised = False
    for x in set(domains[xi]):
        if all(x == y for y in domains[xj]):
            domains[xi].remove(x)
            revised = True
    return revised


# Forward Checking

def forward_check(domains, var, value):
    temp = copy.deepcopy(domains)

    for neighbor in get_neighbors(*var):
        if value in temp[neighbor]:
            temp[neighbor].remove(value)
            if len(temp[neighbor]) == 0:
                return None
    return temp


# Variable Selection (MRV)

def select_unassigned_variable(domains):
    unassigned = [v for v in domains if len(domains[v]) > 1]
    return min(unassigned, key=lambda var: len(domains[var])) if unassigned else None


# Backtracking Search

def backtrack(domains):
    global backtrack_calls, backtrack_failures
    backtrack_calls += 1

    if all(len(domains[v]) == 1 for v in domains):
        return domains

    var = select_unassigned_variable(domains)

    for value in domains[var]:
        new_domains = copy.deepcopy(domains)
        new_domains[var] = {value}

        # Forward Checking
        fc_domains = forward_check(new_domains, var, value)
        if fc_domains is None:
            continue

        # AC-3
        if ac3(fc_domains):
            result = backtrack(fc_domains)
            if result:
                return result

    backtrack_failures += 1
    return None


# Solve Function

def solve(filename):
    global backtrack_calls, backtrack_failures
    backtrack_calls = 0
    backtrack_failures = 0

    board = read_board(filename)
    domains = init_domains(board)

    ac3(domains)
    result = backtrack(domains)

    if result:
        solved = [[next(iter(result[(r, c)])) for c in range(9)] for r in range(9)]
        print("\nSolved Board:")
        print_board(solved)
    else:
        print("No solution found")

    print("\nBacktrack Calls:", backtrack_calls)
    print("Backtrack Failures:", backtrack_failures)

In [15]:
solve("easy.txt")



Solved Board:
7 8 4 9 3 2 1 5 6
6 1 9 4 8 5 3 2 7
2 3 5 1 7 6 4 8 9
5 7 8 2 6 1 9 3 4
3 4 1 8 9 7 5 6 2
9 2 6 5 4 3 8 7 1
4 5 3 7 2 9 6 1 8
8 6 2 3 1 4 7 9 5
1 9 7 6 5 8 2 4 3

Backtrack Calls: 1
Backtrack Failures: 0


In [16]:
solve("medium.txt")



Solved Board:
8 7 5 9 3 6 1 4 2
1 6 9 7 2 4 3 8 5
2 4 3 8 5 1 6 7 9
4 5 2 6 9 7 8 3 1
9 8 6 4 1 3 2 5 7
7 3 1 5 8 2 9 6 4
5 1 7 3 6 9 4 2 8
6 2 8 1 4 5 7 9 3
3 9 4 2 7 8 5 1 6

Backtrack Calls: 2
Backtrack Failures: 0


In [17]:
solve("hard.txt")



Solved Board:
1 5 2 3 4 6 8 9 7
4 3 7 1 8 9 6 5 2
6 8 9 5 7 2 3 1 4
8 2 1 6 3 7 9 4 5
5 4 3 8 9 1 7 2 6
9 7 6 4 2 5 1 8 3
7 9 8 2 5 3 4 6 1
3 6 5 9 1 4 2 7 8
2 1 4 7 6 8 5 3 9

Backtrack Calls: 13
Backtrack Failures: 8


In [18]:
solve("veryhard.txt")


Solved Board:
4 3 1 8 6 7 9 2 5
6 5 2 4 9 1 3 8 7
8 9 7 5 3 2 1 6 4
3 8 4 9 7 6 5 1 2
5 1 9 2 8 4 7 3 6
2 7 6 3 1 5 8 4 9
9 4 3 7 2 8 6 5 1
7 6 5 1 4 3 2 9 8
1 2 8 6 5 9 4 7 3

Backtrack Calls: 53
Backtrack Failures: 40


### Approach
This project implements a Sudoku solver using Constraint Satisfaction Problem (CSP) techniques.
The solver combines:

Backtracking Search : explores possible values

Forward Checking : removes invalid values early

AC-3 Algorithm : enforces arc consistency to reduce domains

Each cell is treated as a variable with domain {1–9}, and constraints ensure:

No repetition in rows

No repetition in columns

No repetition in 3×3 grids
### Analysis
##### Easy Board:
Very few backtracking calls due to strong constraint propagation by AC-3.
##### Medium Board:
Moderate increase in backtracking as ambiguity increases.
##### Hard Board:
Requires significant search; forward checking helps reduce failures.
##### Very Hard Board:
Maximum backtracking and failures due to weak initial constraints and deeper search tree.

Overall, AC-3 and forward checking significantly reduce search space, but harder puzzles still rely heavily on backtracking.